# ESMGAT Training Pipeline — Google Colab (Tesla T4 GPU)

This notebook trains the **GAT graph link predictor** (`GATLinkPredictor`) for the TransGraph-PPI project on Google Colab with free NVIDIA T4 GPU acceleration.

### Invariants Preserved:
- Uses the **exact same STRING graph** (19,431 nodes, ~2.2M edges) and 483-dim node features (480 ESM-2 + 3 topological).
- Uses the **exact same train/validation split** (`train.csv`, `val.csv`).
- Uses the **exact same decoder head**, optimizer (`AdamW`), scheduler (`CosineAnnealingLR`), and focal loss as GraphSAGE.
- `test.csv` is **never touched** during training.
- Outputs are saved strictly into `ESMGAT/weights/` and `ESMGAT/checkpoints/`.
- The stable GraphSAGE checkpoints and models are **completely untouched**.

## 1. Verify NVIDIA T4 GPU Allocation

In [ ]:
!nvidia-smi

## 2. Setup Working Directory
Navigate to your uploaded or cloned `Majorproject` repository directory.

In [ ]:
import os
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Majorproject

# If cloned directly into /content:
%cd /content/Majorproject
!git checkout gat-exp
!pwd

## 3. Install Dependencies
Install PyTorch Geometric and required packages.

In [ ]:
!pip install torch-geometric==2.7.0 xgboost==3.2.0 scikit-learn==1.7.1

## 4. Run Pre-Flight Sanity & Memory Safety Check
This runs the 11-point verification on the real STRING graph and measures VRAM allocation.

In [ ]:
!python ESMGAT/training/train_gat_colab.py --dry_run_only

## 5. Full GAT Training with Early Stopping & Platt Calibration
Runs full training (max 100 epochs, patience 15). Early stopping monitors `val.csv` BCE loss.
Upon completion, Platt calibration is fitted on validation logits and saved.

In [ ]:
!python ESMGAT/training/train_gat_colab.py --epochs 100 --patience 15 --lr 0.001

## 6. Download Artifacts to Local Project
Download the trained GAT weights and calibrator to place into `ESMGAT/weights/` on your local machine.

In [ ]:
from google.colab import files

# Download GAT best model weights (~4.8 MB)
files.download('ESMGAT/weights/gat_model_best.pth')

# Download GAT Platt calibrator (~80 bytes)
files.download('ESMGAT/weights/gat_calibrator.json')

# Download validation metrics and report
files.download('ESMGAT/results/gat_val_metrics.json')
files.download('ESMGAT/results/gat_calibration.json')